In [1]:
import os
import torch
from datasets import load_dataset, DatasetDict, Audio
from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2SeqWithPadding
)
import evaluate
import librosa

# =================================================================================
# 1. КОНФИГУРАЦИЯ: Настройте эти параметры под свою задачу
# =================================================================================

# --- ИЗМЕНЕНИЕ: Указываем путь к JSON-файлу ---
JSON_DATA_PATH = "my_dataset.json" 

# Название базовой модели Whisper с Hugging Face Hub
MODEL_NAME = "openai/whisper-small"

# Язык и задача
LANGUAGE = "russian"
TASK = "transcribe"

# Путь для сохранения дообученной модели
OUTPUT_DIR = "./whisper-small-finetuned-from-json"

# Параметры обучения (остаются такими же)
TRAINING_ARGS = {
    "per_device_train_batch_size": 8,
    "gradient_accumulation_steps": 2,
    "learning_rate": 1e-5,
    "warmup_steps": 50,
    "num_train_epochs": 3,
    "evaluation_strategy": "epoch",
    "save_strategy": "epoch",
    "fp16": True,
    "logging_steps": 25,
    "load_best_model_at_end": True,
    "metric_for_best_model": "wer",
    "greater_is_better": False,
    "push_to_hub": False,
}

# =================================================================================
# 2. ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ (ИЗ JSON)
# =================================================================================

print("Шаг 2: Загрузка и подготовка данных из JSON...")

# --- ИЗМЕНЕНИЕ: Загружаем датасет из JSON ---
# `data_files` указывает на наш файл. `field='train'` говорит, что все данные из файла
# будут в одном сплите 'train'.
dataset = load_dataset("json", data_files=JSON_DATA_PATH, split="train")

# --- ИЗМЕНЕНИЕ: Стандартизируем названия колонок ---
# Переименовываем колонки из нашего JSON в те, которые ожидает скрипт.
# Если ваши ключи в JSON другие, измените "audio_path" и "text".
dataset = dataset.rename_column("audio_path", "audio")
dataset = dataset.rename_column("text", "transcription")

# --- ИЗМЕНЕНИЕ: Загружаем аудиоданные ---
# `cast_column` превращает колонку с путями в настоящую аудио-колонку.
# Библиотека `datasets` будет автоматически загружать и ресемплировать аудио "на лету".
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

# Разделяем на обучающую и тестовую выборки (90% / 10%)
dataset = dataset.train_test_split(test_size=0.1)

# Создаем процессор
processor = WhisperProcessor.from_pretrained(MODEL_NAME, language=LANGUAGE, task=TASK)

# Функция подготовки данных (остается без изменений!)
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
    batch["labels"] = processor.tokenizer(batch["transcription"]).input_ids
    return batch

# Применяем функцию ко всему датасету
print("Обработка датасета...")
dataset = dataset.map(prepare_dataset, remove_columns=dataset.column_names["train"], num_proc=os.cpu_count())

# =================================================================================
# 3. НАСТРОЙКА МОДЕЛИ И ТРЕНЕРА (БЕЗ ИЗМЕНЕНИЙ)
# =================================================================================

print("Шаг 3: Настройка модели и тренера...")
data_collator = DataCollatorForSeq2SeqWithPadding(processor=processor, model=None, padding=True)
metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)
    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language=LANGUAGE, task=TASK)
training_args = Seq2SeqTrainingArguments(output_dir=OUTPUT_DIR, **TRAINING_ARGS)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

# =================================================================================
# 4. ОБУЧЕНИЕ МОДЕЛИ (БЕЗ ИЗМЕНЕНИЙ)
# =================================================================================

print("Шаг 4: Запуск дообучения...")
trainer.train()
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"Модель сохранена в папку: {OUTPUT_DIR}")

# =================================================================================
# 5. ПРОВЕРКА РЕЗУЛЬТАТА (ИНФЕРЕНС) (БЕЗ ИЗМЕНЕНИЙ)
# =================================================================================

print("\nШаг 5: Проверка работы дообученной модели...")
finetuned_model = WhisperForConditionalGeneration.from_pretrained(OUTPUT_DIR)
finetuned_processor = WhisperProcessor.from_pretrained(OUTPUT_DIR)
device = "cuda" if torch.cuda.is_available() else "cpu"
finetuned_model.to(device)
print(f"Модель загружена на устройство: {device}")

def transcribe_audio(audio_path):
    speech_array, _ = librosa.load(audio_path, sr=16000)
    input_features = finetuned_processor(speech_array, sampling_rate=16000, return_tensors="pt").input_features.to(device)
    with torch.no_grad():
        predicted_ids = finetuned_model.generate(input_features)
    transcription = finetuned_processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    return transcription

test_sample = dataset["test"][0]
test_audio_path = test_sample["audio"]["path"]

print(f"\nТранскрибируем тестовый файл: {test_audio_path}")
result_text = transcribe_audio(test_audio_path)

print("-" * 50)
print(f"Оригинальный текст: {test_sample['transcription']}")
print(f"Результат модели  : {result_text}")
print("-" * 50)

c:\Work\Whisper_or_no_Whisper\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: cannot import name 'DataCollatorForSeq2SeqWithPadding' from 'transformers' (c:\Work\Whisper_or_no_Whisper\.venv\Lib\site-packages\transformers\__init__.py)